In [ ]:
import tensorflow as tf
import pandas as pd
from tensorflow import keras
import numpy as np
import matplotlib.pyplot as plt

print(tf.__version__)


In [ ]:
# get data files
!wget https://cdn.freecodecamp.org/project-data/sms/train-data.tsv
!wget https://cdn.freecodecamp.org/project-data/sms/valid-data.tsv

train_file_path = "train-data.tsv"
test_file_path = "valid-data.tsv"

In [ ]:
train_df = pd.read_csv(train_file_path, sep='\t', header=None, names=['label', 'message'])
test_df  = pd.read_csv(test_file_path,  sep='\t', header=None, names=['label', 'message'])

train_df['label'] = (train_df['label'] == 'spam').astype(int)
test_df['label']  = (test_df['label']  == 'spam').astype(int)

train_messages = train_df['message'].values
train_labels   = train_df['label'].values
test_messages  = test_df['message'].values
test_labels    = test_df['label'].values

In [ ]:
VOCAB_SIZE = 1000
MAX_LEN    = 100

vectorize_layer = keras.layers.TextVectorization(
    max_tokens=VOCAB_SIZE,
    output_mode='int',
    output_sequence_length=MAX_LEN
)
vectorize_layer.adapt(train_messages)

X_train = vectorize_layer(train_messages)
X_test  = vectorize_layer(test_messages)

# ── Class Weights ─────────────────────────────────────────────────────────────
spam_ratio = train_labels.mean()
class_weight = {
    0: 1.0,
    1: (1.0 - spam_ratio) / spam_ratio
}
print("Class weights:", class_weight)

# ── Model (no mask_zero, use_cudnn=False) ─────────────────────────────────────
model = keras.Sequential([
    keras.layers.Embedding(VOCAB_SIZE, 32),          # mask_zero removed
    keras.layers.Bidirectional(
        keras.layers.LSTM(32, use_cudnn=False)        # cuDNN disabled
    ),
    keras.layers.Dense(32, activation='relu'),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

# ── Train ─────────────────────────────────────────────────────────────────────
history = model.fit(
    X_train, train_labels,
    epochs=30,
    batch_size=32,
    validation_data=(X_test, test_labels),
    class_weight=class_weight,
    verbose=1
)

# ── Evaluate ──────────────────────────────────────────────────────────────────
loss, acc = model.evaluate(X_test, test_labels, verbose=2)
print(f"Test accuracy: {acc:.4f}")



In [ ]:
# function to predict messages based on model
# (should return list containing prediction and label, ex. [0.008318834938108921, 'ham'])
def predict_message(pred_text):
    vectorized = vectorize_layer(tf.expand_dims(pred_text, axis=0))
    probability = model.predict(vectorized, verbose=0)[0][0]
    label = 'spam' if probability >= 0.5 else 'ham'
    return [float(probability), label]

pred_text = "how are you doing today?"
prediction = predict_message(pred_text)
print(prediction)

In [ ]:
# Run this cell to test your function and model. Do not modify contents.
def test_predictions():
  test_messages = ["how are you doing today",
                   "sale today! to stop texts call 98912460324",
                   "i dont want to go. can we try it a different day? available sat",
                   "our new mobile video service is live. just install on your phone to start watching.",
                   "you have won £1000 cash! call to claim your prize.",
                   "i'll bring it tomorrow. don't forget the milk.",
                   "wow, is your arm alright. that happened to me one time too"
                  ]

  test_answers = ["ham", "spam", "ham", "spam", "spam", "ham", "ham"]
  passed = True

  for msg, ans in zip(test_messages, test_answers):
    prediction = predict_message(msg)
    if prediction[1] != ans:
      passed = False

  if passed:
    print("You passed the challenge. Great job!")
  else:
    print("You haven't passed yet. Keep trying.")

test_predictions()
